In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('darkgrid')

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [6]:
import os

os.listdir("/content")

['.config', '.ipynb_checkpoints', 'Data', 'sample_data']

In [7]:
df = pd.read_csv("./Data/IMDB Dataset.csv")

In [10]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [9]:
df.sample(5)

,review,sentiment
3948,This mess starts off with a real tank running ...,negative
24018,This film has got so much in it. Prehistoric s...,negative
8717,Incarcerated train robber near Yuma breaks fre...,negative
49859,I watched this movie out of a lack of anything...,positive
25995,remember back when this movie was made by robe...,positive


In [8]:
df_pos = df[df['sentiment']=='positive'][:5000]
df_neg = df[df['sentiment']=='negative'][:5000]

df_reviews = pd.concat([df_pos, df_neg ])

In [11]:
from sklearn.model_selection import train_test_split


In [12]:
train,test = train_test_split(df_reviews,test_size =0.33,random_state=42)

In [13]:
train_x, train_y = train['review'], train['sentiment']
test_x, test_y = test['review'], test['sentiment']

In [14]:
train_y.value_counts()

,count
sentiment,
negative,3378
positive,3322


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [26]:
tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=10000
)

train_x_vector = tfidf.fit_transform(train_x)

test_x_vector = tfidf.transform(test_x)

In [17]:
train_x.shape

(6700,)

In [18]:
train_x_vector.shape

(6700, 44107)

In [19]:
type(train_x_vector)

scipy.sparse._csr.csr_matrix

In [20]:
primera_resenia = pd.DataFrame.sparse.from_spmatrix(train_x_vector,
                                  index=train_x.index,
                                  columns=tfidf.get_feature_names_out()).iloc[0]

In [21]:
primera_resenia

,6746
00,0
000,0
007,0
00am,0
00s,0
...,...
ísnt,0
île,0
önsjön,0
über,0


In [22]:
train_x.iloc[0]

"I happened to rent this movie with my sister in hopes of watching a great entertaining movie, that was humorous, however my expectations were let down. This movie was beyond disgusting and revolting for a PG-13 movie, this should have been rated R for the many mature references that went on in this movie. I wouldn't recommend allowing a 13 year old teen see this.<br /><br />Even if no one under the age of 17 is watching this movie, beware of a truly stupid movie, there's no humor in the movie, just a bunch of disgusting sexual references including a small touch of pedophilia, something that shouldn't even be joked about. <br /><br />I would like to know what happened to PG-13 movies, that were actually safe for actual a 13 year old? This is beyond a deplorable movie and should be re-rated."

In [23]:
primera_resenia[primera_resenia != 0]

,6746
13,0.45849
17,0.12824
actual,0.091601
actually,0.061461
age,0.088765
allowing,0.12824
beware,0.143046
br,0.124945
bunch,0.093128
deplorable,0.168137


In [24]:
train_x.iloc[0]

"I happened to rent this movie with my sister in hopes of watching a great entertaining movie, that was humorous, however my expectations were let down. This movie was beyond disgusting and revolting for a PG-13 movie, this should have been rated R for the many mature references that went on in this movie. I wouldn't recommend allowing a 13 year old teen see this.<br /><br />Even if no one under the age of 17 is watching this movie, beware of a truly stupid movie, there's no humor in the movie, just a bunch of disgusting sexual references including a small touch of pedophilia, something that shouldn't even be joked about. <br /><br />I would like to know what happened to PG-13 movies, that were actually safe for actual a 13 year old? This is beyond a deplorable movie and should be re-rated."

In [27]:
from sklearn.svm import SVC
svc = SVC(kernel='linear')
svc.fit(train_x_vector, train_y)

SVC(kernel='linear')

In [28]:
print(svc.predict(tfidf.transform(['A good movie'])))
print(svc.predict(tfidf.transform(['An excellent movie'])))
print(svc.predict(tfidf.transform(['I did not like this movie at all I gave this movie away'])))

['positive']
['positive']
['negative']


In [29]:
print(svc.score(test_x_vector, test_y))

0.8721212121212121


In [30]:
from sklearn.metrics import f1_score

f1_score(test_y,svc.predict(test_x_vector),
          labels = ['positive','negative'],average=None)

array([0.87544274, 0.86861768])

In [31]:
from sklearn.metrics import classification_report

print(classification_report(test_y,
                            svc.predict(test_x_vector),
                            labels = ['positive','negative']))

              precision    recall  f1-score   support

    positive       0.87      0.88      0.88      1678
    negative       0.88      0.86      0.87      1622

    accuracy                           0.87      3300
   macro avg       0.87      0.87      0.87      3300
weighted avg       0.87      0.87      0.87      3300



In [32]:
from sklearn.metrics import confusion_matrix

conf_mat = confusion_matrix(test_y,
                           svc.predict(test_x_vector),
                           labels = ['positive', 'negative'])
conf_mat

array([[1483,  195],
       [ 227, 1395]])

In [33]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [34]:
log_model = LogisticRegression(max_iter=1000)

log_model.fit(train_x_vector, train_y)

log_pred = log_model.predict(test_x_vector)

print("Accuracy Logistic Regression:")
print(accuracy_score(test_y, log_pred))

print("\nReporte:")
print(classification_report(test_y, log_pred))

Accuracy Logistic Regression:
0.8727272727272727

Reporte:
              precision    recall  f1-score   support

    negative       0.88      0.86      0.87      1622
    positive       0.87      0.88      0.88      1678

    accuracy                           0.87      3300
   macro avg       0.87      0.87      0.87      3300
weighted avg       0.87      0.87      0.87      3300



In [35]:
train_dense = train_x_vector.toarray()
test_dense = test_x_vector.toarray()

In [36]:
nb_model = GaussianNB()

nb_model.fit(train_dense, train_y)

nb_pred = nb_model.predict(test_dense)

print("Accuracy GaussianNB:")
print(accuracy_score(test_y, nb_pred))

print("\nReporte:")
print(classification_report(test_y, nb_pred))

Accuracy GaussianNB:
0.7142424242424242

Reporte:
              precision    recall  f1-score   support

    negative       0.69      0.77      0.73      1622
    positive       0.75      0.66      0.70      1678

    accuracy                           0.71      3300
   macro avg       0.72      0.72      0.71      3300
weighted avg       0.72      0.71      0.71      3300



In [37]:
tree_model = DecisionTreeClassifier(random_state=42)

tree_model.fit(train_x_vector, train_y)

tree_pred = tree_model.predict(test_x_vector)

print("Accuracy Decision Tree:")
print(accuracy_score(test_y, tree_pred))

print("\nReporte:")
print(classification_report(test_y, tree_pred))

Accuracy Decision Tree:
0.7218181818181818

Reporte:
              precision    recall  f1-score   support

    negative       0.71      0.73      0.72      1622
    positive       0.73      0.71      0.72      1678

    accuracy                           0.72      3300
   macro avg       0.72      0.72      0.72      3300
weighted avg       0.72      0.72      0.72      3300



In [38]:
comparacion = pd.DataFrame({
    "Modelo": [
        "SVM",
        "Logistic Regression",
        "GaussianNB",
        "Decision Tree"
    ],

    "Accuracy": [
        svc.score(test_x_vector,test_y),
        accuracy_score(test_y, log_pred),
        accuracy_score(test_y, nb_pred),
        accuracy_score(test_y, tree_pred)
    ],

    "F1 Score": [
        f1_score(test_y, svc.predict(test_x_vector), average="weighted"),
        f1_score(test_y, log_pred, average="weighted"),
        f1_score(test_y, nb_pred, average="weighted"),
        f1_score(test_y, tree_pred, average="weighted")
    ]
})

comparacion

,Modelo,Accuracy,F1 Score
0,SVM,0.872121,0.872088
1,Logistic Regression,0.872727,0.872707
2,GaussianNB,0.714242,0.713494
3,Decision Tree,0.721818,0.721835
